# Clinical Triage Model Comparison

This notebook prepares the synthetic FedMML triage data and compares local
MedGemma and MedSigLIP models. The real MIMIC-IV-ED fine-tuning experiment is
in `medgemma_finetuning_pipeline.ipynb`.

In [ ]:
from pathlib import Path
import os

import pandas as pd

def find_project_root(start: Path, marker: str = "fedmml_ed_triage_dataset.csv") -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {marker}. Start Jupyter from the repository root "
        "or place the CSV beside this notebook."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

os.chdir(PROJECT_ROOT)
RAW_CSV = PROJECT_ROOT / "fedmml_ed_triage_dataset.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_CSV


## Load data

In [ ]:
raw_df = pd.read_csv(RAW_CSV)
raw_df.shape


In [ ]:
raw_df.head()


In [ ]:
raw_df.columns.tolist()


## Select columns

In [ ]:
FEATURE_COLUMNS = [
    "age",
    "sex",
    "chief_complaint",
    "clinical_notes",
    "systolic_bp",
    "diastolic_bp",
    "heart_rate",
    "respiratory_rate",
    "temperature",
    "spo2",
    "pain_score",
]
LABEL_COLUMN = "esi_level"

required_columns = FEATURE_COLUMNS + [LABEL_COLUMN]
missing_required = [column for column in required_columns if column not in raw_df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

raw_df[required_columns].head()


## Build the modeling table

`esi_level` is the target. Identifiers, timestamps, site fields, and laboratory
values are excluded. Rows with missing model inputs are dropped.

In [ ]:
modeling_df = raw_df[FEATURE_COLUMNS + [LABEL_COLUMN]].dropna(axis=0, how="any").copy()
y = modeling_df[LABEL_COLUMN].astype("int64").rename("label_esi_level")
X = modeling_df[FEATURE_COLUMNS].copy()

print("Raw shape:", raw_df.shape)
print("Rows after dropping missing values:", len(modeling_df))
print("Rows removed:", len(raw_df) - len(modeling_df))
print("Feature shape:", X.shape)
print("Label shape:", y.shape)
print("Kept feature columns:", FEATURE_COLUMNS)
print("Label column:", LABEL_COLUMN)


In [ ]:
assert X.columns.tolist() == FEATURE_COLUMNS
assert LABEL_COLUMN not in X.columns
assert len(X) == len(y)

X.head()


## Label distribution

ESI 1 is the highest-acuity level; ESI 5 is the lowest.

In [ ]:
label_distribution = y.value_counts().sort_index().rename_axis("esi_level").reset_index(name="count")
label_distribution["fraction"] = label_distribution["count"] / len(y)
label_distribution


## Missing values

In [ ]:
missingness = X.isna().sum().sort_values(ascending=False).rename("missing_count").reset_index()
missingness.columns = ["column", "missing_count"]
missingness["missing_fraction"] = missingness["missing_count"] / len(X)
missingness.head(20)


## Save processed data

In [ ]:
features_path = PROCESSED_DIR / "clinical_triage_features.csv"
labels_path = PROCESSED_DIR / "clinical_triage_labels.csv"
combined_path = PROCESSED_DIR / "clinical_triage_cleaned_with_label.csv"

X.to_csv(features_path, index=False)
y.to_frame().to_csv(labels_path, index=False)
pd.concat([X, y], axis=1).to_csv(combined_path, index=False)

print(features_path)
print(labels_path)
print(combined_path)


In [ ]:
check_features = pd.read_csv(features_path, nrows=5)
check_labels = pd.read_csv(labels_path, nrows=5)
check_combined = pd.read_csv(combined_path, nrows=5)

print("features columns match selected set:", check_features.columns.tolist() == FEATURE_COLUMNS)
print("features contains esi_level:", LABEL_COLUMN in check_features.columns)
print("labels columns:", check_labels.columns.tolist())
print("combined columns tail:", check_combined.columns[-5:].tolist())

check_features.head()


## Prompt

Each patient row is serialized as JSON. MedGemma returns
`predicted_esi_level` as 1-5 or `uncertain`.

In [ ]:
from clinical_triage_utils import (
    PROMPT_FEATURE_COLUMNS,
    build_esi_prompt,
    case_json_to_retrieval_text,
    embed_texts_medsiglip,
    empty_result_counter,
    is_valid_model_input,
    generate_medgemma_4b_prediction,
    generate_text_model_prediction,
    row_to_case_json,
    score_prediction,
)

PROMPT_TEMPLATE_PATH = PROJECT_ROOT / "prompts" / "esi_prediction_prompt.txt"
PROMPT_TEMPLATE_PATH.exists(), PROMPT_TEMPLATE_PATH

## Runtime

Set `FORCE_CPU = True` to disable CUDA. Otherwise, the notebook uses a CUDA
GPU when one is available.

In [ ]:
import sys
import torch

# Set True to disable CUDA.
FORCE_CPU = False

GPU_AVAILABLE = torch.cuda.is_available()
USE_GPU = GPU_AVAILABLE and not FORCE_CPU

MAX_RANDOM_ENTRIES = 100
RANDOM_STATE = 765

MODEL_DEVICE = "cuda" if USE_GPU else "cpu"
MODEL_DTYPE = torch.bfloat16 if USE_GPU else torch.float32
MODEL_DEVICE_MAP = "auto" if USE_GPU else None

print(f"Python: {sys.executable}")
print(f"PyTorch: {torch.__version__}; CUDA build: {torch.version.cuda}")
print(f"GPU available: {GPU_AVAILABLE}")
print(f"Using device: {MODEL_DEVICE}")
print(f"Model dtype: {MODEL_DTYPE}")
print(f"Device map: {MODEL_DEVICE_MAP}")
if GPU_AVAILABLE:
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.memory_allocated(0) / 1024**2:.1f} MiB allocated, {torch.cuda.memory_reserved(0) / 1024**2:.1f} MiB reserved")


## Model evaluation

All models use fixed random samples. MedSigLIP is a nearest-neighbor retrieval
baseline over text embeddings; it is not fine-tuned here.

In [ ]:
from IPython.display import display
from tqdm.auto import tqdm

RESULT_COLUMNS = ["model", "row", "actual", "prediction", "status", "model_output", "details"]

def show_run_header(model_id, rows_count):
    print(f"Model: {model_id}")
    print(f"Evaluation rows: {rows_count}")
    print(f"Device setting: {MODEL_DEVICE}")

def show_results(counts, rows):
    display(pd.DataFrame([dict(counts)]))
    return pd.DataFrame(rows, columns=RESULT_COLUMNS)

eval_df = pd.read_csv(PROCESSED_DIR / "clinical_triage_cleaned_with_label.csv")
eval_rows = eval_df.sample(
    n=min(MAX_RANDOM_ENTRIES, len(eval_df)),
    random_state=RANDOM_STATE,
).reset_index(drop=True)

print(f"Shared evaluation sample: {len(eval_rows)} rows")

## MedGemma 4B

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

MEDGEMMA_4B_MODEL_ID = "google/medgemma-1.5-4b-it"

show_run_header(MEDGEMMA_4B_MODEL_ID, len(eval_rows))

processor_4b = AutoProcessor.from_pretrained(MEDGEMMA_4B_MODEL_ID, local_files_only=True)
model_4b = AutoModelForImageTextToText.from_pretrained(
    MEDGEMMA_4B_MODEL_ID,
    local_files_only=True,
    torch_dtype=MODEL_DTYPE,
    device_map=MODEL_DEVICE_MAP,
)
model_4b.eval()
print("Model placement:", getattr(model_4b, "hf_device_map", "single-device"))
print("First parameter device:", next(model_4b.parameters()).device)
if torch.cuda.is_available():
    print(f"CUDA memory after load: {torch.cuda.memory_allocated(0) / 1024**2:.1f} MiB allocated, {torch.cuda.memory_reserved(0) / 1024**2:.1f} MiB reserved")

counts_4b = empty_result_counter()
rows_4b = []
for row_index, row in tqdm(eval_rows.iterrows(), total=len(eval_rows), desc="MedGemma 4B cases", unit="case"):
    actual = int(row["label_esi_level"])
    if not is_valid_model_input(row):
        counts_4b["invalid_input"] += 1
        rows_4b.append({
            "model": MEDGEMMA_4B_MODEL_ID,
            "row": row_index,
            "actual": actual,
            "prediction": None,
            "status": "invalid_input",
            "model_output": None,
            "details": {},
        })
        continue
    try:
        prompt = build_esi_prompt(row_to_case_json(row), PROMPT_TEMPLATE_PATH)
        prediction, raw_response = generate_medgemma_4b_prediction(prompt, processor_4b, model_4b)
        output = {"predicted_esi_level": prediction}
    except Exception as exc:
        prediction = "invalid_output"
        raw_response = ""
        output = f"ERROR: {exc}"
    status = score_prediction(prediction, actual, counts_4b)
    rows_4b.append({
        "model": MEDGEMMA_4B_MODEL_ID,
        "row": row_index,
        "actual": actual,
        "prediction": prediction,
        "status": status,
        "model_output": output,
        "details": {"raw_response": raw_response},
    })

show_results(counts_4b, rows_4b)

## MedGemma 27B

In [ ]:
# Disabled by default because the 27B model exceeds typical laptop memory.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

RUN_27B_TEXT = False
MEDGEMMA_27B_TEXT_MODEL_ID = "google/medgemma-27b-text-it"
EVAL_SAMPLE_SIZE_27B = min(MAX_RANDOM_ENTRIES, len(eval_rows))

eval_rows_27b = eval_rows.head(EVAL_SAMPLE_SIZE_27B).copy()
show_run_header(MEDGEMMA_27B_TEXT_MODEL_ID, len(eval_rows_27b))

counts_27b = empty_result_counter()
rows_27b = []

if not RUN_27B_TEXT:
    print("Skipped: RUN_27B_TEXT is False.")
else:
    tokenizer_27b = AutoTokenizer.from_pretrained(MEDGEMMA_27B_TEXT_MODEL_ID, local_files_only=True)
    model_27b = AutoModelForCausalLM.from_pretrained(
        MEDGEMMA_27B_TEXT_MODEL_ID,
        local_files_only=True,
        torch_dtype=torch.float32,
        device_map=None,
    )
    model_27b.eval()
    print("Model placement: single-device")
    print("First parameter device:", next(model_27b.parameters()).device)

    for row_index, row in tqdm(eval_rows_27b.iterrows(), total=len(eval_rows_27b), desc="MedGemma 27B cases", unit="case"):
        actual = int(row["label_esi_level"])
        if not is_valid_model_input(row):
            counts_27b["invalid_input"] += 1
            rows_27b.append({
                "model": MEDGEMMA_27B_TEXT_MODEL_ID,
                "row": row_index,
                "actual": actual,
                "prediction": None,
                "status": "invalid_input",
                "model_output": None,
                "details": {},
            })
            continue
        try:
            prompt = build_esi_prompt(row_to_case_json(row), PROMPT_TEMPLATE_PATH)
            prediction, raw_response = generate_text_model_prediction(prompt, tokenizer_27b, model_27b)
            output = {"predicted_esi_level": prediction}
        except Exception as exc:
            prediction = "invalid_output"
            raw_response = ""
            output = f"ERROR: {exc}"
        status = score_prediction(prediction, actual, counts_27b)
        rows_27b.append({
            "model": MEDGEMMA_27B_TEXT_MODEL_ID,
            "row": row_index,
            "actual": actual,
            "prediction": prediction,
            "status": status,
            "model_output": output,
            "details": {"raw_response": raw_response},
        })

show_results(counts_27b, rows_27b)

## MedSigLIP

In [ ]:
import numpy as np
from transformers import AutoModel, AutoProcessor

MEDSIGLIP_MODEL_ID = "google/medsiglip-448"
REFERENCE_SIZE = 200
MEDSIGLIP_EVAL_SIZE = min(MAX_RANDOM_ENTRIES, len(eval_rows))
MEDSIGLIP_BATCH_SIZE = 8
TOP_K = 5

medsiglip_df = eval_df.sample(
    n=min(REFERENCE_SIZE + MEDSIGLIP_EVAL_SIZE, len(eval_df)),
    random_state=RANDOM_STATE,
).reset_index(drop=True)
reference_df = medsiglip_df.iloc[: min(REFERENCE_SIZE, len(medsiglip_df) - 1)].copy()
query_df = medsiglip_df.iloc[len(reference_df) : len(reference_df) + MEDSIGLIP_EVAL_SIZE].copy()

show_run_header(MEDSIGLIP_MODEL_ID, len(query_df))
print(f"Reference rows: {len(reference_df)}")
print(f"Embedding batch size: {MEDSIGLIP_BATCH_SIZE}")

processor_siglip = AutoProcessor.from_pretrained(MEDSIGLIP_MODEL_ID, local_files_only=True)
model_siglip = AutoModel.from_pretrained(MEDSIGLIP_MODEL_ID, local_files_only=True)
model_siglip.to(MODEL_DEVICE)
model_siglip.eval()
print("Model placement: single-device")
print("First parameter device:", next(model_siglip.parameters()).device)
if torch.cuda.is_available():
    print(f"CUDA memory after load: {torch.cuda.memory_allocated(0) / 1024**2:.1f} MiB allocated, {torch.cuda.memory_reserved(0) / 1024**2:.1f} MiB reserved")

reference_texts = [case_json_to_retrieval_text(row_to_case_json(row)) for _, row in reference_df.iterrows()]
query_texts = [case_json_to_retrieval_text(row_to_case_json(row)) for _, row in query_df.iterrows()]
reference_embeddings = embed_texts_medsiglip(
    reference_texts,
    processor_siglip,
    model_siglip,
    batch_size=MEDSIGLIP_BATCH_SIZE,
    desc="MedSigLIP reference embeddings",
)
query_embeddings = embed_texts_medsiglip(
    query_texts,
    processor_siglip,
    model_siglip,
    batch_size=MEDSIGLIP_BATCH_SIZE,
    desc="MedSigLIP query embeddings",
)
if torch.cuda.is_available():
    print(f"CUDA memory after embeddings: {torch.cuda.memory_allocated(0) / 1024**2:.1f} MiB allocated, {torch.cuda.memory_reserved(0) / 1024**2:.1f} MiB reserved")

counts_siglip = empty_result_counter()
rows_siglip = []
reference_labels = reference_df["label_esi_level"].astype(int).to_numpy()

for row_offset, (_, row) in tqdm(enumerate(query_df.iterrows()), total=len(query_df), desc="MedSigLIP cases", unit="case"):
    actual = int(row["label_esi_level"])
    if not is_valid_model_input(row):
        counts_siglip["invalid_input"] += 1
        rows_siglip.append({
            "model": MEDSIGLIP_MODEL_ID,
            "row": int(row_offset),
            "actual": actual,
            "prediction": None,
            "status": "invalid_input",
            "model_output": None,
            "details": {},
        })
        continue
    scores = reference_embeddings @ query_embeddings[row_offset]
    top_indices = np.argsort(-scores)[:TOP_K]
    top_labels = reference_labels[top_indices]
    values, vote_counts = np.unique(top_labels, return_counts=True)
    prediction = int(values[np.argmax(vote_counts)])
    output = {"predicted_esi_level": prediction}
    status = score_prediction(prediction, actual, counts_siglip)
    rows_siglip.append({
        "model": MEDSIGLIP_MODEL_ID,
        "row": int(row_offset),
        "actual": actual,
        "prediction": prediction,
        "status": status,
        "model_output": output,
        "details": {
            "top_k_labels": top_labels.tolist(),
            "top_score": float(scores[top_indices[0]]),
        },
    })

show_results(counts_siglip, rows_siglip)